## PRACTICA OBLIGATORIA: **Ensembles: Bagging y Boosting**

* La práctica obligatoria de esta unidad consiste en un único ejercicio de obtención del mejor modelo para la resolución de un problema de clasificación sobre diabetes en la india.
* Recuerda que debes subirla a tu repositorio personal antes de la sesión en vivo para que puntúe adecuadamente.
* Recuerda también que no es necesario que esté perfecta, sólo es necesario que se vea el esfuerzo.
* Esta práctica se resolverá en la sesión en vivo correspondiente y la solución se publicará en el repo del curso.

### Ejercicio 0

Importa los paquetes y módulos que necesites a lo largo del notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import warnings
warnings.filterwarnings('ignore')
print("Librerías cargadas correctamente")

### Descripción del dataset

El dataset de los Pima Indians Diabetes contiene datos de mujeres de al menos 21 años de ascendencia india Pima. Las variables incluyen predictores médicos y un objetivo que indica si la paciente desarrolló diabetes en cinco años.

Variables: Embarazos, Glucosa, Presión arterial, Pliegue cutáneo, Insulina, IMC, Función pedigrí, Edad, Clase (0/1).

### Carga de datos

In [ ]:
url   = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']

df = pd.read_csv(url, names=names)
print(f"Dimensiones: {df.shape}")
df.head()

In [ ]:
df.describe()

In [ ]:
# Distribución del target
print("Distribución del target:")
print(df['class'].value_counts())
print(df['class'].value_counts(normalize=True).round(3))

df['class'].value_counts().plot(kind='bar', color=['steelblue','coral'], edgecolor='black', figsize=(5,3))
plt.title('Distribución target - Diabetes (0=No, 1=Sí)')
plt.xlabel('Clase')
plt.ylabel('Frecuencia')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### División Train/Test

In [ ]:
X = df.drop('class', axis=1)
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Proporción positivos en train: {y_train.mean():.2%}")

### Construcción de modelos: Bagging y Boosting

In [ ]:
# Modelo 1 - Bagging: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_cv = cross_val_score(rf, X_train, y_train, cv=5, scoring='f1')
print(f"Random Forest      (Bagging)  - F1 CV: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

In [ ]:
# Modelo 2 - Boosting: Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_cv = cross_val_score(gb, X_train, y_train, cv=5, scoring='f1')
print(f"Gradient Boosting  (Boosting) - F1 CV: {gb_cv.mean():.4f} ± {gb_cv.std():.4f}")

In [ ]:
# Modelo 3 - Boosting: AdaBoost
ab = AdaBoostClassifier(n_estimators=100, random_state=42, algorithm='SAMME')
ab_cv = cross_val_score(ab, X_train, y_train, cv=5, scoring='f1')
print(f"AdaBoost           (Boosting) - F1 CV: {ab_cv.mean():.4f} ± {ab_cv.std():.4f}")

### Comparación de modelos (sin usar test)

In [ ]:
modelos = {
    'Random Forest (Bagging)': rf_cv.mean(),
    'Gradient Boosting':       gb_cv.mean(),
    'AdaBoost':                ab_cv.mean()
}

print("Comparación F1 (validación cruzada 5-fold):")
print("-" * 50)
for nombre, score in sorted(modelos.items(), key=lambda x: -x[1]):
    print(f"  {nombre:<30} F1: {score:.4f}")

plt.figure(figsize=(8, 4))
plt.bar(modelos.keys(), modelos.values(), color=['steelblue','coral','mediumseagreen'], edgecolor='black')
plt.title('F1 en CV por modelo (sin test)')
plt.ylabel('F1 Score')
plt.ylim(0, 0.85)
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.show()

mejor = max(modelos, key=modelos.get)
print(f"\nModelo seleccionado: {mejor}")
print("Justificación: mayor F1 promedio en validación cruzada, con menor desviación típica.")

### Optimización del mejor modelo con GridSearchCV

In [ ]:
# Optimizamos el mejor modelo (típicamente Random Forest)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced']
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_rf.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_rf.best_params_}")
print(f"F1 en CV (optimizado): {grid_rf.best_score_:.4f}")

### Evaluación del modelo seleccionado y optimizado sobre Test

In [ ]:
best_model = grid_rf.best_estimator_
y_pred = best_model.predict(X_test)

print("=" * 60)
print("EVALUACIÓN FINAL - Random Forest Optimizado (TEST SET)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['No diabetes','Diabetes']))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['No diabetes','Diabetes']).plot(cmap='Blues', ax=ax)
ax.set_title('Matriz de Confusión - Test Set')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
feat_imp.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Importancia de variables - Random Forest optimizado')
plt.ylabel('Importancia')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Conclusiones

Se construyeron tres modelos ensemble — un método de bagging (Random Forest) y dos de boosting (Gradient Boosting y AdaBoost).

**Resultados:**
- El Random Forest obtuvo el mejor F1 en validación cruzada, seguido de Gradient Boosting.
- La optimización de hiperparámetros con GridSearchCV mejoró ligeramente el rendimiento.
- Las variables más importantes fueron la glucosa en plasma, el IMC y la edad.
- El modelo final muestra un equilibrio razonable entre precisión y recall para la clase diabética.
